# 02 — Feature extraction and group comparison in each condition (Layer 1)

Input: the output folder of `01_preprocessing.ipynb`. The extraction and test functions are in
`src/eeg_features.py`, shared with `04_stability_sensitivity.ipynb`.

1. **Analytic sample.** Subjects present in all three conditions (EC, EO, TASK) with at
   least `MIN_WINDOWS` valid 2-s windows and `MIN_BLOCKS` 10-s blocks of consecutive
   windows in every condition.
2. **Equalization.** The number of 2-s windows per subject and condition is capped at the
   10th percentile of the available windows (`n_target`), and the number of 10-s blocks at
   `n_long_target`, by random selection without replacement with deterministic seeds per
   subject and condition.
3. **Features.** 1,216 features per subject and condition in six domains: spectral (408),
   time-frequency (480), complexity (96), band ratios (96), hemispheric asymmetry (56) and
   functional connectivity (80).
4. **Layer 1.** HC versus MDD in each condition: two-sided Mann-Whitney test with
   Benjamini-Hochberg FDR over the 1,216 features of each condition (the only detection
   criterion of the study), Cliff's delta with a bootstrap 95% CI, and Hedges' g.
5. **Descriptive detection pattern.** Conditions in which each feature is significant. This
   pattern is descriptive only; condition dependence is tested formally in
   `03_condition_dependence.ipynb`.
6. **Rejection diagnostic.** Within each group and condition, Spearman correlation between the
   window rejection rate and each feature, with FDR over the features.

**Outputs (in `RESULTS_DIR`):**

| File | Content |
|---|---|
| `features_<COND>.csv` | One row per subject: `subject`, `group` and the 1,216 features |
| `stats_<COND>.csv` | Layer 1 statistics per feature |
| `sampling_log_<COND>.csv` | Seeds and selected windows and blocks per subject |
| `layer1_summary.csv` | Per feature: detection, Cliff's delta and FDR p in each condition, detection pattern |
| `layer1_domain_table.csv` | Significant features by domain and condition |
| `subject_window_counts.csv` | Windows available and used per subject and condition |
| `rejection_diagnostic.csv` | Rejection rates by group and summary of the correlations per group and condition |
| `rejection_feature_correlations.csv` | Spearman rho, p and FDR q of every feature, group and condition |
| `parameters_features.json` | All parameters of `src/eeg_features.py` |
| `cwt_frequencies.csv` | Requested and effective CWT frequencies, scales and edge trimming (Table S2) |
| `connectivity_pairs.csv` | Channel pairs of each connectivity summary |

## Setup

In [ ]:
# Google Colab only: install dependencies and mount Google Drive.
import sys
if 'google.colab' in sys.modules:
    !pip install -q PyWavelets statsmodels
    from google.colab import drive
    drive.mount('/content/drive')

In [ ]:
import os
import sys

REPO_DIR = '..'   # repository root (on Colab: the folder where the repository was cloned or copied)
sys.path.insert(0, os.path.join(REPO_DIR, 'src'))
import json
import warnings

import numpy as np
import pandas as pd
import pywt
import scipy
import statsmodels
from scipy.stats import mannwhitneyu, spearmanr
from statsmodels.stats.multitest import multipletests

import eeg_features as F
from eeg_features import CONDITIONS

warnings.filterwarnings('ignore')
print(f'Python {sys.version.split()[0]}')
for module in (np, scipy, pd, pywt, statsmodels):
    print(f'{module.__name__:12s} {module.__version__}')

## Configuration

In [ ]:
PREPROCESSED_DIR = 'path/to/preprocessed'   # OUTPUT_DIR of 01_preprocessing.ipynb
RESULTS_DIR = 'path/to/results'
os.makedirs(RESULTS_DIR, exist_ok=True)

F.check_channel_order(PREPROCESSED_DIR)   # arrays are stored in F.CHANNEL_NAMES order
for name, value in F.parameters().items():
    if name not in ('REGIONS', 'ASYMMETRY_PAIRS', 'LEFT_CH', 'RIGHT_CH'):
        print(f'{name:22s} {value}')

## Methods

The functions are in `src/eeg_features.py`, in the order of the sections below. Sites: 19
channels and 5 regional averages (frontal, central, temporal, parietal, occipital); the
regional value is the mean of the channel values, and the subject value is the mean across
windows or blocks.

### Spectral domain (2-s windows)

Welch PSD (Hann window of 256 samples, 50% overlap, 1-Hz resolution). Band intervals are
$[f_{lo}, f_{hi})$, except the last band, which includes $f_{hi}$. Absolute power is the sum
of PSD bins times $\Delta f$; relative power is divided by the total 1–45 Hz power.

### Time-frequency domain (10-s blocks)

Complex Morlet CWT at 5 frequencies per band (centers of 5 equal sub-intervals). One period
of the lowest band frequency is trimmed at each edge of the block. Mean, SD, coefficient of
variation and IQR of the band power time course are computed within each block and
averaged across blocks.

### Complexity domain

Sample entropy, permutation entropy and Higuchi fractal dimension in 2-s windows; Hurst
exponent (rescaled range) in 10-s blocks.

### Functional connectivity (10-s blocks)

Magnitude-squared coherence (Welch, 512-sample segments, 50% overlap) and phase-locking
value (linear-phase FIR band-pass, Hilbert phase). Matrices are averaged across blocks and
summarized over channel pairs: global, intra-regional, inter-hemispheric and
fronto-posterior.

### Hemispheric asymmetry and band ratios (from the subject-mean absolute power)

### Analytic sample and equalization

10-s blocks are built only from 5 windows whose original indices (saved by
`01_preprocessing.ipynb`) are strictly consecutive, so windows separated by a rejected
segment are never concatenated. Blocks do not overlap and each window is used at most once.

### Layer 1 statistics

Two-sided Mann-Whitney test per feature, Benjamini-Hochberg FDR over the features of each
condition. Cliff's delta: positive = MDD > HC; 95% percentile bootstrap CI (2,000 resamples
of subjects within each group), reported as a measure of precision only. Hedges' g with
ddof = 1 and small-sample correction.

### Descriptive detection pattern

The conditions in which each feature reaches FDR significance. A feature significant in
one condition and not in another is **not** evidence that the group difference depends on
the condition; this is tested formally in `03_condition_dependence.ipynb`.

### Rejection diagnostic

In the analytic sample, Spearman correlation, within each group and condition, between the
proportion of rejected windows (`processing_log.json` of `01_preprocessing.ipynb`) and each of
the 1,216 features, with Benjamini-Hochberg FDR over the features of each group and condition.
The rejection rates of the two groups are compared with a two-sided Mann-Whitney test. A group
with fewer than 3 distinct rejection rates in a condition is skipped.

In [ ]:
def rejection_diagnostic(condition, features, rejection):
    d = features.assign(rejection=[rejection[(condition, s)] for s in features.subject])
    names = [col for col in features.columns if col not in ('subject', 'group')]
    correlations, summary = [], []
    for group in ('control', 'mdd'):
        g = d[d.group == group]
        row = {'condition': condition, 'group': group, 'n': len(g), 'rejection_median': g.rejection.median(),
               'n_distinct_rejection': g.rejection.nunique()}
        if g.rejection.nunique() >= 3:
            res = [spearmanr(g.rejection, g[f], nan_policy='omit') for f in names]
            rho = np.array([r[0] for r in res])
            p = np.array([r[1] for r in res], float)
            q = np.full(len(p), np.nan)
            ok = np.isfinite(p)
            q[ok] = multipletests(p[ok], method='fdr_bh')[1]
            correlations.append(pd.DataFrame({'condition': condition, 'group': group, 'feature': names,
                                              'rho': rho, 'p': p, 'q': q}))
            significant = [f for f, qq in zip(names, q) if qq < F.ALPHA_FDR]
            row.update(n_p_below_005=int((p < 0.05).sum()), n_expected=round(0.05 * ok.sum()),
                       n_fdr=len(significant), min_q=float(np.nanmin(q)),
                       families=', '.join(f'{k} ({v})' for k, v in
                                          pd.Series([F.family_of(f) for f in significant]).value_counts().items()))
        summary.append(row)
    hc, mdd = d.loc[d.group == 'control', 'rejection'], d.loc[d.group == 'mdd', 'rejection']
    for row in summary:
        row['p_rejection_groups'] = mannwhitneyu(hc, mdd, alternative='two-sided').pvalue
    return pd.concat(correlations, ignore_index=True), summary

## Run

### 1. Analytic sample

In [ ]:
sample, n_target, n_long_target, n_win, n_blk = F.select_sample(PREPROCESSED_DIR)

### 2. Features and Layer 1 in each condition

In [ ]:
stats_by_cond, features_by_cond = {}, {}
for cond in CONDITIONS:
    print(f"\n{'=' * 70}\n{cond}\n{'-' * 70}")
    stats_by_cond[cond], features_by_cond[cond] = F.run_condition(
        PREPROCESSED_DIR, cond, sample, n_target, n_long_target, save_dir=RESULTS_DIR)

### 3. Detection pattern and table by domain

In [ ]:
summary = F.layer1_summary(stats_by_cond)
summary.to_csv(os.path.join(RESULTS_DIR, 'layer1_summary.csv'), index=False)

table = F.domain_table(summary)
table.to_csv(os.path.join(RESULTS_DIR, 'layer1_domain_table.csv'))
print('Significant features (FDR) by domain and condition:')
print(table.to_string())

print('\nDetection pattern (features significant in at least one condition):')
print(summary.loc[summary.n_conditions_detected > 0, 'detection_pattern'].value_counts().to_string())

### 4. Windows per subject (supplementary table)

In [ ]:
counts = pd.DataFrame({
    'subject': sorted(sample, key=lambda s: (s.split('_')[0], int(s.split('_')[1]))),
})
counts['group'] = counts.subject.str.split('_').str[0]
for cond in CONDITIONS:
    log = pd.read_csv(os.path.join(RESULTS_DIR, f'sampling_log_{cond}.csv')).set_index('subject')
    counts[f'n_windows_available_{cond}'] = counts.subject.map(log['n_windows_available'])
    counts[f'n_windows_used_{cond}'] = counts.subject.map(log['n_windows_used'])
    counts[f'n_blocks_available_{cond}'] = counts.subject.map(log['n_blocks_available'])
counts.to_csv(os.path.join(RESULTS_DIR, 'subject_window_counts.csv'), index=False)
print(counts.describe().loc[['min', '50%', 'max']].round(0).to_string())

### 5. Rejection diagnostic (analytic sample)

In [ ]:
with open(os.path.join(PREPROCESSED_DIR, 'processing_log.json'), encoding='utf-8') as fh:
    rejection = {(lg['condition'], lg['subject_id']): lg['rejection_rate']
                 for lg in json.load(fh) if lg.get('success')}

correlations, summary = [], []
for cond in CONDITIONS:
    corr, summ = rejection_diagnostic(cond, features_by_cond[cond], rejection)
    correlations.append(corr)
    summary += summ
pd.concat(correlations, ignore_index=True).to_csv(
    os.path.join(RESULTS_DIR, 'rejection_feature_correlations.csv'), index=False)
summary = pd.DataFrame(summary)
summary.to_csv(os.path.join(RESULTS_DIR, 'rejection_diagnostic.csv'), index=False)
print(summary.round(3).to_string(index=False))

### 6. Parameter tables

In [ ]:
with open(os.path.join(RESULTS_DIR, 'parameters_features.json'), 'w') as fh:
    json.dump(F.parameters(), fh, indent=2)
cwt = F.cwt_frequency_table()
cwt.to_csv(os.path.join(RESULTS_DIR, 'cwt_frequencies.csv'), index=False)
pairs = F.connectivity_pair_table()
pairs.to_csv(os.path.join(RESULTS_DIR, 'connectivity_pairs.csv'), index=False)
print(cwt.round(3).to_string(index=False))
print('\nChannel pairs per connectivity summary:', pairs.groupby('summary').size().to_dict())